# VQA Múa Lân — Train & Eval (A1, A2, A3)

Huấn luyện và đánh giá **3 cấu hình decoder** trên cùng pipeline:

| Run | Decoder       | Norm      | FFN        | Ghi chú |
|-----|---------------|-----------|------------|---------|
| **A1** | LSTM        | —         | —          | Baseline rời (RNN cổ điển) |
| **A2** | Transformer | LayerNorm | VanillaFFN | Transformer cổ điển |
| **A3** | Transformer | RMSNorm   | SwiGLU     | Modern stack (LLaMA / Gemma / Qwen) |

Encoder cố định: SigLIP2-B/16 (layer −2, frozen) + PhoBERT-v2 (mean 4 last layers, frozen) + Cross-Attention Fusion.
Loss: `CrossEntropy(ignore_index=PAD, label_smoothing=0.1)`.

> **Precompute pretrained**: SigLIP2 + PhoBERT-v2 sẽ được tải về 1 lần và lưu vào working space (`/kaggle/working/pretrained/`). Các lần chạy sau load từ disk, không cần kết nối HuggingFace.

## 0. Kaggle Setup

In [1]:
!git clone https://github.com/KhanhNamYeh/qa-domain

Cloning into 'qa-domain'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 64 (delta 20), reused 52 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 199.06 KiB | 3.62 MiB/s, done.
Resolving deltas: 100% (20/20), done.


In [2]:
!pwd

/kaggle/working


In [3]:
!ls

qa-domain


## 1. Setup

In [4]:
import os, sys, time, json, random
import numpy as np
import torch
import matplotlib.pyplot as plt

# PROJECT_ROOT = os.path.abspath('.')
PROJECT_ROOT = os.path.abspath('/kaggle/working/qa-domain')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import ModelConfig, TrainConfig
from src.build  import (build_tokenizer_and_processor, resolve_special_ids,
                        build_loaders, build_model)
from src.training import Trainer, Evaluator
from src.metrics  import ExactMatch, BLEUScore, METEORScore

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('project_root =', PROJECT_ROOT)
print('device       =', DEVICE)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

project_root = /kaggle/working/qa-domain
device       = cuda


## 1.5. Precompute pretrained weights (chạy 1 lần)

Tải SigLIP2 + PhoBERT-v2 từ HuggingFace, gọi `save_pretrained()` để lưu thẳng vào working space. Các lần chạy sau (kể cả khi không có internet) sẽ load từ thư mục local — bỏ qua hoàn toàn HuggingFace.

In [ ]:
from transformers import AutoModel, AutoTokenizer, AutoImageProcessor

# Working space cho weight + cache pretrained.
WORKSPACE_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else PROJECT_ROOT
PRETRAINED_DIR = os.path.join(WORKSPACE_DIR, 'pretrained')
SIGLIP_LOCAL  = os.path.join(PRETRAINED_DIR, 'siglip2-base-patch16-224')
PHOBERT_LOCAL = os.path.join(PRETRAINED_DIR, 'phobert-base-v2')
os.makedirs(PRETRAINED_DIR, exist_ok=True)

SIGLIP_HF  = 'google/siglip2-base-patch16-224'
PHOBERT_HF = 'vinai/phobert-base-v2'

def _has_local(path):
    """Coi là cached khi có config.json trong thư mục."""
    return os.path.isdir(path) and os.path.exists(os.path.join(path, 'config.json'))

# --- SigLIP2 ---
if _has_local(SIGLIP_LOCAL):
    print(f'[SigLIP2] cached at {SIGLIP_LOCAL}')
else:
    print(f'[SigLIP2] downloading from HF -> {SIGLIP_LOCAL}')
    _m = AutoModel.from_pretrained(SIGLIP_HF)
    _m.save_pretrained(SIGLIP_LOCAL)
    AutoImageProcessor.from_pretrained(SIGLIP_HF).save_pretrained(SIGLIP_LOCAL)
    del _m

# --- PhoBERT-v2 ---
if _has_local(PHOBERT_LOCAL):
    print(f'[PhoBERT-v2] cached at {PHOBERT_LOCAL}')
else:
    print(f'[PhoBERT-v2] downloading from HF -> {PHOBERT_LOCAL}')
    _m = AutoModel.from_pretrained(PHOBERT_HF)
    _m.save_pretrained(PHOBERT_LOCAL)
    AutoTokenizer.from_pretrained(PHOBERT_HF, use_fast=False).save_pretrained(PHOBERT_LOCAL)
    del _m

print('SIGLIP_LOCAL  =', SIGLIP_LOCAL)
print('PHOBERT_LOCAL =', PHOBERT_LOCAL)

## 1.6. Precompute encoded features (chạy 1 lần)

Encode toàn bộ train/val/test qua SigLIP2 + PhoBERT-v2 (đã frozen) ngay bây giờ, lưu thành `.npy` (fp16) trong `<WORKSPACE>/cache/{train,val,test}/`.

**Lợi ích chính:**
- Khi train, model chỉ chạy `Linear(768→512) + Fusion + Decoder` — không gọi SigLIP/PhoBERT mỗi batch nữa.
- Một epoch giảm ~10× thời gian (đa số chi phí trước đây nằm ở 2 backbone).
- Restart kernel → load lại trong vài giây, không cần GPU cho phần encode.

In [ ]:
from transformers import AutoTokenizer, AutoImageProcessor
from src.data import precompute_split, is_cached

CACHE_ROOT = os.path.join(WORKSPACE_DIR, 'cache')
os.makedirs(CACHE_ROOT, exist_ok=True)

# Đường dẫn JSON + ảnh — cần thiết để encode
TRAIN_JSON = os.path.join(PROJECT_ROOT, 'qa_data', 'train.json')
VAL_JSON   = os.path.join(PROJECT_ROOT, 'qa_data', 'val.json')
TEST_JSON  = os.path.join(PROJECT_ROOT, 'qa_data', 'test.json')
IMAGE_ROOT = os.path.join('/kaggle/input/datasets/namkhn/qa-domain', 'images') \
             if os.path.isdir('/kaggle/input/datasets/namkhn/qa-domain/images') \
             else os.path.join(PROJECT_ROOT, 'images')

MAX_Q_LEN, MAX_A_LEN = 32, 64

# Tokenizer + image processor đọc từ thư mục local đã save_pretrained ở cell 1.5
tokenizer = AutoTokenizer.from_pretrained(PHOBERT_LOCAL, use_fast=False)
image_processor = AutoImageProcessor.from_pretrained(SIGLIP_LOCAL)

for split, jp in [('train', TRAIN_JSON), ('val', VAL_JSON), ('test', TEST_JSON)]:
    out = os.path.join(CACHE_ROOT, split)
    if is_cached(out):
        print(f'[{split}] cached at {out}')
        continue
    print(f'[{split}] precomputing -> {out}')
    precompute_split(
        json_path=jp,
        tokenizer=tokenizer,
        image_processor=image_processor,
        siglip_path=SIGLIP_LOCAL,
        phobert_path=PHOBERT_LOCAL,
        image_root=IMAGE_ROOT,
        out_dir=out,
        max_question_len=MAX_Q_LEN,
        max_answer_len=MAX_A_LEN,
        batch_size=32,
        device=DEVICE,
        save_dtype='float16',
    )
print('CACHE_ROOT =', CACHE_ROOT)

## 2. Cached DataLoaders (đọc từ npy precomputed)

In [ ]:
from src.build import build_cached_loaders

# Probe ModelConfig chỉ để có vocab_size / pad_id / bos_id / eos_id từ tokenizer.
_probe_cfg = ModelConfig(
    image_encoder_name=SIGLIP_LOCAL,
    text_encoder_name=PHOBERT_LOCAL,
    max_answer_len=MAX_A_LEN,
)
_probe_cfg = resolve_special_ids(tokenizer, _probe_cfg)

BATCH_SIZE = 64           # có thể tăng vì không còn chạy 2 backbone trong train loop
EPOCHS     = 12
LR         = 3e-4

CKPT_DIR = os.path.join(WORKSPACE_DIR, 'checkpoints')
HIST_DIR = os.path.join(WORKSPACE_DIR, 'logs')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(HIST_DIR, exist_ok=True)

shared_train_cfg = TrainConfig(
    train_json=TRAIN_JSON, val_json=VAL_JSON, test_json=TEST_JSON,
    image_root=IMAGE_ROOT,
    batch_size=BATCH_SIZE, epochs=EPOCHS, lr=LR, num_workers=2,
    ckpt_dir=CKPT_DIR, log_dir=HIST_DIR,
    max_question_len=MAX_Q_LEN, max_answer_len=MAX_A_LEN,
    eval_every=1, save_every=EPOCHS,
)
train_loader, val_loader, test_loader = build_cached_loaders(CACHE_ROOT, shared_train_cfg)
print(f'train={len(train_loader.dataset)}  val={len(val_loader.dataset)}  test={len(test_loader.dataset)}')
print(f'sample img_feat shape:', train_loader.dataset[0]['img_feat'].shape)
print(f'sample txt_feat shape:', train_loader.dataset[0]['txt_feat'].shape)

## 3. Trainer-with-history

Bọc lại `Trainer` để lưu `train_loss` và mọi metric val theo từng epoch — phục vụ vẽ biểu đồ line.

In [6]:
class TrainerWithHistory(Trainer):
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self.history = {'train_loss': [], 'val': {}}
    def fit(self):
        cfg = self.train_cfg
        for epoch in range(cfg.epochs):
            t0 = time.time()
            tl = self.train_one_epoch(epoch)
            self.history['train_loss'].append(tl)
            log = f'epoch={epoch+1}/{cfg.epochs} train_loss={tl:.4f} time={time.time()-t0:.1f}s'
            if self.evaluator is not None and self.val_loader is not None:
                res = self.evaluator.evaluate(self.model, self.val_loader)
                for k, v in res.items():
                    self.history['val'].setdefault(k, []).append(v)
                log += '  ' + ' '.join(f'val_{k}={v:.4f}' for k, v in res.items())
            self._log(log)
        self._save_ckpt('final')
        return self.history

In [ ]:
from src.build import build_cached_model

def run_config(tag, decoder_type, norm_type='layernorm', ffn_type='vanilla'):
    """Train cached VQA model. `norm_type`/`ffn_type` chỉ ảnh hưởng khi
    decoder_type='transformer'."""
    print(f'\n========== {tag} ==========')
    mcfg = ModelConfig(
        image_encoder_name=SIGLIP_LOCAL,
        text_encoder_name=PHOBERT_LOCAL,
        decoder_type=decoder_type, norm_type=norm_type, ffn_type=ffn_type,
        max_answer_len=MAX_A_LEN,
    )
    mcfg = resolve_special_ids(tokenizer, mcfg)
    tcfg = TrainConfig(
        train_json=TRAIN_JSON, val_json=VAL_JSON, test_json=TEST_JSON,
        image_root=IMAGE_ROOT,
        batch_size=BATCH_SIZE, epochs=EPOCHS, lr=LR, num_workers=2,
        ckpt_dir=CKPT_DIR, log_dir=HIST_DIR,
        max_question_len=MAX_Q_LEN, max_answer_len=MAX_A_LEN,
        eval_every=1, save_every=EPOCHS, run_name=tag,
    )
    model = build_cached_model(mcfg)
    metrics = [ExactMatch(), BLEUScore(), METEORScore()]
    evaluator = Evaluator(
        tokenizer=tokenizer, metrics=metrics,
        bos_id=mcfg.bos_id, eos_id=mcfg.eos_id, pad_id=mcfg.pad_id,
        max_len=mcfg.max_answer_len, beam_size=1,
    )
    trainer = TrainerWithHistory(
        model=model, train_loader=train_loader, val_loader=val_loader,
        train_cfg=tcfg, model_cfg=mcfg, evaluator=evaluator,
    )
    history = trainer.fit()
    return history, trainer, evaluator, model

## 4. Train A1 — LSTM

In [ ]:
hist_A1, trainer_A1, evaluator_A1, model_A1 = run_config(
    'A1_lstm', decoder_type='lstm')

## 5. Train A2 — Transformer + LayerNorm + VanillaFFN

In [ ]:
hist_A2, trainer_A2, evaluator_A2, model_A2 = run_config(
    'A2_transformer_vanilla', decoder_type='transformer', norm_type='layernorm', ffn_type='vanilla')

## 6. Train A3 — Transformer + RMSNorm + SwiGLU

In [ ]:
hist_A3, trainer_A3, evaluator_A3, model_A3 = run_config(
    'A3_transformer_modern', decoder_type='transformer', norm_type='rmsnorm', ffn_type='swiglu')

## 7. So sánh — biểu đồ line

In [ ]:
def plot_compare(histories, keys, title_prefix=''):
    n = len(keys)
    fig, axes = plt.subplots(1, n, figsize=(5.2*n, 4))
    if n == 1: axes = [axes]
    for ax, key in zip(axes, keys):
        for tag, h in histories.items():
            ys = h['train_loss'] if key == 'train_loss' else h['val'].get(key, [])
            xs = list(range(1, len(ys)+1))
            ax.plot(xs, ys, marker='o', label=tag)
        ax.set_title(f'{title_prefix}{key}')
        ax.set_xlabel('epoch'); ax.grid(True, alpha=0.3); ax.legend()
    plt.tight_layout(); plt.show()

histories = {
    'A1_lstm':                hist_A1,
    'A2_transformer_vanilla': hist_A2,
    'A3_transformer_modern':  hist_A3,
}
plot_compare(histories, ['train_loss'])
plot_compare(histories, ['exact_match', 'bleu', 'meteor'], title_prefix='val_')

## 8. Đánh giá cuối trên test split (Exact Match, BLEU, METEOR)

In [ ]:
test_results = {}
for tag, model in [('A1_lstm', model_A1),
                   ('A2_transformer_vanilla', model_A2),
                   ('A3_transformer_modern',  model_A3)]:
    metrics = [ExactMatch(), BLEUScore(), METEORScore()]
    ev = Evaluator(
        tokenizer=tokenizer, metrics=metrics,
        bos_id=_probe_cfg.bos_id, eos_id=_probe_cfg.eos_id, pad_id=_probe_cfg.pad_id,
        max_len=_probe_cfg.max_answer_len, beam_size=1,
    )
    test_results[tag] = ev.evaluate(model, test_loader)
    print(tag, test_results[tag])

## 9. Lưu lại history + test results

In [ ]:
os.makedirs(HIST_DIR, exist_ok=True)
summary_path = os.path.join(HIST_DIR, 'train_eval_A_history.json')
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump({'history': histories, 'test': test_results}, f, ensure_ascii=False, indent=2)
print('saved ->', summary_path)